# Citation Auditor and Guardrails

**PartnerLens Copilot — Final Submission Notebook**

This notebook uses repository-relative paths and does not require Google Drive. Run it from a cloned copy of the `partnerlens-copilot` repository after installing `requirements.txt`.

## Environment setup

From a terminal opened at the repository root, install dependencies once:

```bash
python -m pip install -r requirements.txt
```

The next cell locates the repository automatically when Jupyter is started from the repository or its `notebooks` folder.

In [1]:
from pathlib import Path
import importlib.util
import os
import sys


def locate_repository_root() -> Path:
    """Locate the PartnerLens repository without relying on Google Drive."""
    current = Path.cwd().resolve()
    candidates = [
        current,
        *current.parents,
        Path.home() / "partnerlens-copilot",
        Path("/content/partnerlens-copilot"),
    ]

    visited = set()
    for candidate in candidates:
        try:
            candidate = candidate.expanduser().resolve()
        except OSError:
            continue
        if candidate in visited:
            continue
        visited.add(candidate)
        if (
            (candidate / "README.md").is_file()
            and (candidate / "src").is_dir()
            and (candidate / "data").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "PartnerLens repository root was not found. Start Jupyter from inside "
        "the partnerlens-copilot repository, or clone the repository to "
        "~/partnerlens-copilot or /content/partnerlens-copilot."
    )


REPO_ROOT = locate_repository_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

required_modules = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sqlparse": "sqlparse",
    "openpyxl": "openpyxl",
}
missing_packages = [
    package_name
    for module_name, package_name in required_modules.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    raise ModuleNotFoundError(
        "Missing required packages: "
        + ", ".join(missing_packages)
        + ". From the repository root run: "
        + f"{sys.executable} -m pip install -r requirements.txt"
    )

RAW_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
CONFIG_DIR = REPO_ROOT / "configs"
NOTEBOOK_OUTPUT_DIR = REPO_ROOT / "artifacts" / "notebook_outputs"
EVALUATION_OUTPUT_DIR = REPO_ROOT / "artifacts" / "evaluation"
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"Python executable: {sys.executable}")
print("Notebook environment is ready.")

Repository root: C:\Users\nafee\partnerlens-copilot
Python executable: C:\Users\nafee\AppData\Local\Programs\Python\Python312\python.exe
Notebook environment is ready.


## 1. Citation-auditor components

This notebook tests the production functions in `src/answer_generator.py` and `src/citation_auditor.py`. It verifies record-level citations, source-table coverage, grounding language, fabricated citations, and zero-result handling.

In [2]:
import re
import pandas as pd
from IPython.display import Markdown, display

from src.answer_generator import generate_answer
import importlib
import inspect
import src.citation_auditor as citation_auditor_module

citation_auditor_module = importlib.reload(
    citation_auditor_module
)

audit_citations = citation_auditor_module.audit_citations

print(
    "Loaded citation auditor from:",
    citation_auditor_module.__file__,
)

print(
    "audit_citations signature:",
    inspect.signature(audit_citations),
)

sample_records = [
    {
        "partner_id": "P000101",
        "partner_name": "Desert Peak Payments",
        "industry_vertical": "Technology",
        "state": "AZ",
        "txn_growth_pct": 28.4,
        "payment_volume_usd": 1_250_000.0,
        "net_revenue_usd": 42_500.0,
    },
    {
        "partner_id": "P000202",
        "partner_name": "Canyon Commerce",
        "industry_vertical": "Retail",
        "state": "AZ",
        "txn_growth_pct": 23.1,
        "payment_volume_usd": 980_000.0,
        "net_revenue_usd": 31_250.0,
    },
]

source_tables = ("partner_current_preview",)
question = "Show me partners in Arizona with transaction growth above 20%"

grounded_answer = generate_answer(
    user_question=question,
    result_records=sample_records,
    source_tables=source_tables,
    max_records=2,
)

display(Markdown(grounded_answer))

Loaded citation auditor from: C:\Users\nafee\partnerlens-copilot\src\citation_auditor.py
audit_citations signature: (answer: str, result_records: list[dict] | None, source_tables: list[str] | tuple[str, ...] | None, max_records: int = 5) -> dict


Based on the validated SQL query results, I found 2 matching partner record(s).

### 1. Desert Peak Payments
- **Industry Vertical:** Technology
- **State:** AZ
- **Transaction Growth:** 28.40%
- **Payment Volume:** $1,250,000.00
- **Net Revenue:** $42,500.00
- **Citation:** SQLite partner record `P000101`; fields returned by the validated SQL query.

### 2. Canyon Commerce
- **Industry Vertical:** Retail
- **State:** AZ
- **Transaction Growth:** 23.10%
- **Payment Volume:** $980,000.00
- **Net Revenue:** $31,250.00
- **Citation:** SQLite partner record `P000202`; fields returned by the validated SQL query.

### Source and grounding
- **Source tables:** `partner_current_preview`
- **Grounding:** This answer was generated exclusively from the validated SQL query results in the PartnerLens SQLite database.
- **Evidence scope:** Only fields returned by the validated query were used. No external information or unsupported assumptions were added.

In [3]:
from src.citation_auditor import audit_citations
import inspect

actual_signature = inspect.signature(audit_citations)
print(actual_signature)

assert "result_records" in actual_signature.parameters
assert "source_tables" in actual_signature.parameters
assert "max_records" in actual_signature.parameters

print("Citation auditor interface is correct.")

(answer: str, result_records: list[dict] | None, source_tables: list[str] | tuple[str, ...] | None, max_records: int = 5) -> dict
Citation auditor interface is correct.


In [4]:
print("Loaded from:", citation_auditor_module.__file__)
print(
    "Function signature:",
    inspect.signature(citation_auditor_module.audit_citations),
)

Loaded from: C:\Users\nafee\partnerlens-copilot\src\citation_auditor.py
Function signature: (answer: str, result_records: list[dict] | None, source_tables: list[str] | tuple[str, ...] | None, max_records: int = 5) -> dict


In [5]:
grounded_audit = audit_citations(
    answer=grounded_answer,
    result_records=sample_records,
    source_tables=source_tables,
    max_records=2,
)

display(pd.DataFrame([grounded_audit["checks"]]).T.rename(columns={0: "passed"}))
print("Audit passed:", grounded_audit["citation_audit_passed"])
print("Audit score:", grounded_audit["audit_score_pct"])
print("Reason codes:", grounded_audit["reason_codes"])

,passed
answer_present,True
result_metadata_available,True
source_metadata_available,True
source_tables_approved,True
grounding_statement_present,True
source_table_coverage_complete,True
record_citation_present,True
record_citation_coverage_complete,True
no_unexpected_record_citations,True


Audit passed: True
Audit score: 100.0
Reason codes: []


## 2. Guardrail test scenarios

The following scenarios deliberately damage otherwise grounded answers. A strong auditor should pass the valid and zero-result cases while rejecting missing citations, fabricated citations, missing source-table references, and missing grounding language.

In [6]:
answer_without_record_citations = re.sub(
    r"(?im)^.*SQLite\s+partner\s+record.*$",
    "",
    grounded_answer,
)

answer_with_fabricated_citation = grounded_answer.replace(
    "SQLite partner record `P000101`",
    "SQLite partner record `P999999`",
)

answer_without_source_table = grounded_answer.replace(
    "partner_current_preview",
    "source_table_removed",
)

answer_without_grounding_statement = grounded_answer.replace(
    "validated SQL query results",
    "available information",
)

zero_result_answer = generate_answer(
    user_question=question,
    result_records=[],
    source_tables=source_tables,
    max_records=2,
)

scenario_definitions = [
    {
        "scenario": "valid_grounded_answer",
        "answer": grounded_answer,
        "records": sample_records,
        "expected_pass": True,
    },
    {
        "scenario": "missing_record_citations",
        "answer": answer_without_record_citations,
        "records": sample_records,
        "expected_pass": False,
    },
    {
        "scenario": "fabricated_record_citation",
        "answer": answer_with_fabricated_citation,
        "records": sample_records,
        "expected_pass": False,
    },
    {
        "scenario": "missing_source_table",
        "answer": answer_without_source_table,
        "records": sample_records,
        "expected_pass": False,
    },
    {
        "scenario": "missing_grounding_statement",
        "answer": answer_without_grounding_statement,
        "records": sample_records,
        "expected_pass": False,
    },
    {
        "scenario": "valid_zero_result_answer",
        "answer": zero_result_answer,
        "records": [],
        "expected_pass": True,
    },
]

scenario_results = []
for scenario in scenario_definitions:
    audit = audit_citations(
        answer=scenario["answer"],
        result_records=scenario["records"],
        source_tables=source_tables,
        max_records=2,
    )
    actual_pass = audit["citation_audit_passed"]
    scenario_results.append(
        {
            "scenario": scenario["scenario"],
            "expected_pass": scenario["expected_pass"],
            "actual_pass": actual_pass,
            "expectation_met": actual_pass == scenario["expected_pass"],
            "audit_score_pct": audit["audit_score_pct"],
            "reason_codes": ", ".join(audit["reason_codes"]),
            "missing_record_ids": ", ".join(audit["missing_record_ids"]),
            "unexpected_record_ids": ", ".join(audit["unexpected_record_ids"]),
            "missing_source_tables": ", ".join(audit["missing_source_tables"]),
        }
    )

citation_scenarios_df = pd.DataFrame(scenario_results)
display(citation_scenarios_df)

,scenario,expected_pass,actual_pass,expectation_met,audit_score_pct,reason_codes,missing_record_ids,unexpected_record_ids,missing_source_tables
0,valid_grounded_answer,True,True,True,100.0,,,,
1,missing_record_citations,False,False,True,77.8,"record_citations_missing, record_citation_cove...","P000101, P000202",,
2,fabricated_record_citation,False,False,True,77.8,"record_citation_coverage_incomplete, unexpecte...",P000101,P999999,
3,missing_source_table,False,False,True,88.9,source_table_citation_incomplete,,,partner_current_preview
4,missing_grounding_statement,False,False,True,88.9,grounding_statement_missing,,,
5,valid_zero_result_answer,True,True,True,100.0,,,,


## 3. Citation-auditor metrics

In [7]:
negative_cases = citation_scenarios_df[~citation_scenarios_df["expected_pass"]]
positive_cases = citation_scenarios_df[citation_scenarios_df["expected_pass"]]

citation_metrics_df = pd.DataFrame(
    [
        {
            "metric": "scenario_count",
            "value": len(citation_scenarios_df),
        },
        {
            "metric": "scenario_expectation_accuracy_pct",
            "value": round(100 * citation_scenarios_df["expectation_met"].mean(), 2),
        },
        {
            "metric": "valid_answer_pass_rate_pct",
            "value": round(100 * positive_cases["actual_pass"].mean(), 2),
        },
        {
            "metric": "guardrail_detection_rate_pct",
            "value": round(100 * (~negative_cases["actual_pass"]).mean(), 2),
        },
        {
            "metric": "average_audit_score_pct",
            "value": round(citation_scenarios_df["audit_score_pct"].mean(), 2),
        },
    ]
)

scenario_path = NOTEBOOK_OUTPUT_DIR / "citation_auditor_scenarios.csv"
metrics_path = NOTEBOOK_OUTPUT_DIR / "citation_auditor_metrics.csv"
citation_scenarios_df.to_csv(scenario_path, index=False)
citation_metrics_df.to_csv(metrics_path, index=False)

print(f"Saved: {scenario_path.relative_to(REPO_ROOT)}")
print(f"Saved: {metrics_path.relative_to(REPO_ROOT)}")
display(citation_metrics_df)

if not citation_scenarios_df["expectation_met"].all():
    raise AssertionError("One or more citation-auditor scenarios did not meet expectations.")

print("All citation-auditor guardrail scenarios behaved as expected.")

Saved: artifacts\notebook_outputs\citation_auditor_scenarios.csv
Saved: artifacts\notebook_outputs\citation_auditor_metrics.csv


,metric,value
0,scenario_count,6.0
1,scenario_expectation_accuracy_pct,100.0
2,valid_answer_pass_rate_pct,100.0
3,guardrail_detection_rate_pct,100.0
4,average_audit_score_pct,88.9


All citation-auditor guardrail scenarios behaved as expected.


## Final interpretation

A successful run demonstrates that grounded answers pass, fabricated or incomplete citation evidence is detected, and empty query results are handled without unsupported claims. The scenario and metric files should be committed as evaluator-visible evidence.